In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
keras.utils.set_random_seed(42)


In [ ]:
t = np.linspace(0, 6 * np.pi, 300)
y = np.sin(t) + np.random.normal(0, 0.1, 300)

plt.figure(figsize=(12, 4))
plt.plot(t, y, label='Noisy sine wave')
plt.xlabel('Time')
plt.ylabel('Value')
plt.title('Generated Noisy Sine Wave')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
window_size = 10
forecast_steps = 3

X = []
y_target = []

for i in range(len(y) - window_size - forecast_steps + 1):
    X.append(y[i:i + window_size])
    y_target.append(y[i + window_size:i + window_size + forecast_steps])

X = np.array(X)
y_target = np.array(y_target)

X = X.reshape(X.shape[0], X.shape[1], 1)

print(f"X shape: {X.shape}")
print(f"y shape: {y_target.shape}")


In [ ]:
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y_target[:split_idx], y_target[split_idx:]

print(f"Train samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")


In [ ]:
model = keras.Sequential([
    layers.LSTM(64, return_sequences=False, input_shape=(10, 1)),
    layers.Dense(32, activation='relu'),
    layers.Dense(3)
])

model.compile(loss='mse', optimizer='adam', metrics=['mse'])
model.summary()


In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Training History')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
y_pred = model.predict(X_test)

overall_mse = np.mean((y_test - y_pred) ** 2)
print(f"Overall Test MSE: {overall_mse:.6f}")


In [ ]:
mse_step1 = np.mean((y_test[:, 0] - y_pred[:, 0]) ** 2)
mse_step2 = np.mean((y_test[:, 1] - y_pred[:, 1]) ** 2)
mse_step3 = np.mean((y_test[:, 2] - y_pred[:, 2]) ** 2)

print(f"\nStep-wise MSE:")
print(f"Step 1 MSE: {mse_step1:.6f}")
print(f"Step 2 MSE: {mse_step2:.6f}")
print(f"Step 3 MSE: {mse_step3:.6f}")


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for step in range(3):
    axes[step].plot(y_test[:, step], label=f'Actual Step {step + 1}', alpha=0.7)
    axes[step].plot(y_pred[:, step], label=f'Predicted Step {step + 1}', alpha=0.7)
    axes[step].set_xlabel('Sample Index')
    axes[step].set_ylabel('Value')
    axes[step].set_title(
        f'Step {step + 1} Forecast: Actual vs Predicted (MSE: {[mse_step1, mse_step2, mse_step3][step]:.6f})')
    axes[step].legend()
    axes[step].grid(True)

plt.tight_layout()
plt.show()


In [ ]:

model_deep = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(10, 1)),
    layers.LSTM(64, return_sequences=False),
    layers.Dense(32, activation='relu'),
    layers.Dense(3)
])

model_deep.compile(loss='mse', optimizer='adam', metrics=['mse'])
model_deep.summary()


In [ ]:
# Train deeper model
history_deep = model_deep.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
# Plot training history for deeper model
plt.figure(figsize=(10, 4))
plt.plot(history_deep.history['loss'], label='Training Loss (Deep)')
plt.plot(history_deep.history['val_loss'], label='Validation Loss (Deep)')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Training History - Deeper Model')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Evaluate deeper model
y_pred_deep = model_deep.predict(X_test)

# Calculate step-wise MSE for deeper model
mse_step1_deep = np.mean((y_test[:, 0] - y_pred_deep[:, 0]) ** 2)
mse_step2_deep = np.mean((y_test[:, 1] - y_pred_deep[:, 1]) ** 2)
mse_step3_deep = np.mean((y_test[:, 2] - y_pred_deep[:, 2]) ** 2)

print(f"\nDeeper Model Step-wise MSE:")
print(f"Step 1 MSE: {mse_step1_deep:.6f}")
print(f"Step 2 MSE: {mse_step2_deep:.6f}")
print(f"Step 3 MSE: {mse_step3_deep:.6f}")

print(f"\nComparison for Step 3:")
print(f"Original Model Step 3 MSE: {mse_step3:.6f}")
print(f"Deeper Model Step 3 MSE: {mse_step3_deep:.6f}")
print(f"Improvement: {(mse_step3 - mse_step3_deep):.6f} ({((mse_step3 - mse_step3_deep) / mse_step3 * 100):.2f}%)")


In [ ]:
# Plot comparison for all 3 steps
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for step in range(3):
    axes[step].plot(y_test[:, step], label=f'Actual Step {step + 1}', alpha=0.6, linewidth=2)
    axes[step].plot(y_pred[:, step], label=f'Original Model', alpha=0.7)
    axes[step].plot(y_pred_deep[:, step], label=f'Deeper Model', alpha=0.7)
    axes[step].set_xlabel('Sample Index')
    axes[step].set_ylabel('Value')
    axes[step].set_title(f'Step {step + 1} Forecast Comparison')
    axes[step].legend()
    axes[step].grid(True)

plt.tight_layout()
plt.show()